# Exploring the LTEE reads

Questions: how long are the reads, and how does GC content vary between reads? Exploration only: results that matter go into `reports/ltee-report.qmd`.

In [ ]:
# Run from the project folder, like the scripts (lesson 02).
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

samples = pd.read_csv("metadata/samples.tsv", sep="\t")
samples

In [ ]:
# One row per read: sample, read length and GC content (first 5,000 reads of read 1).
# A FASTQ file has four lines per read; the second one is the sequence.
rows = []
for _, s in samples.iterrows():
    with open(s["fastq_1"]) as fh:
        for i, line in enumerate(fh):
            if i == 4 * 5000:
                break
            if i % 4 == 1:
                seq = line.strip()
                gc = 100 * (seq.count("G") + seq.count("C")) / len(seq)
                rows.append({"sample": s["sample"], "length": len(seq), "gc": gc})
reads = pd.DataFrame(rows)
reads.groupby("sample")[["length", "gc"]].describe().round(1)

In [ ]:
#| label: fig-lengths
#| fig-cap: "Read length after trimming, first 5,000 reads of read 1 of each sample."
#| fig-alt: "Histogram of read lengths from 25 to 150 bases. A single tall bar at 150 bases holds about 60 percent of the reads; the rest are spread thinly between 25 and 149 bases."
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(reads["length"], bins=50, color="#2a78d6")
ax.set_xlabel("Read length (bases)")
ax.set_ylabel("Reads")
plt.show()

About 60% of the reads are 150 bases long, the length of the sequencing run. Trimming shortened the others, down to 25 bases, the shortest length that was kept.

In [ ]:
# Share of reads shorter than 100 bases, per sample
(reads.assign(short=reads["length"] < 100)
      .groupby("sample")["short"].mean()
      .mul(100).round(1)
      .rename("% reads < 100 bases"))